In [ ]:
# Cell 1: imports and project-root setup
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the project root by searching upward for a directory that contains `src`."""
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.pca_encoder import PCAEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    train_iql_from_loader,
    save_metrics_json,
)

In [ ]:
# Cell 2: experiment configuration
METHOD = "pca"

def noise_tag(noise_dim, noise_scale, noise_type):
    """Build a compact tag used in checkpoint and metrics directories."""
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG = f"seed_{SEED}"

CKPT_DIR = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_DIR = OBS_STATS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OBS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("CKPT_DIR:", CKPT_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("OBS_DIR:", OBS_DIR)

# Dataset and Dataloader

In [ ]:
# Cell 3: dataset and dataloader
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)

# Infer dimensions directly from the dataset tensors.
state_dim = dataset.noisy_obs.shape[1]
action_dim = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim

# PCA projects down to true_state_dim components (matches clean state size).
LATENT_DIM = true_state_dim

# Save observation normalization statistics for later evaluation.
np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)

print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim (= true_state_dim):", LATENT_DIM)
print("Saved obs stats:", OBS_DIR / "obs_stats.npz")

# Fit PCA Encoder

PCA is fit on the full noisy offline dataset (no privileged information, no neural network).
The explained variance ratio measures how much signal is retained after projecting to
`true_state_dim` components. For structured nonlinear noise, this is expected to be low,
demonstrating the limitation of linear denoising baselines.

In [ ]:
# Cell 4: fit PCA on the full offline dataset
torch.manual_seed(SEED)
np.random.seed(SEED)

pca_encoder = PCAEncoder(n_components=LATENT_DIM)
pca_encoder.fit(dataset.noisy_obs)

# Save PCA components for reproducibility and later analysis.
pca_ckpt = CKPT_DIR / "pca_components.npz"
pca_encoder.save(pca_ckpt)
print("Saved PCA components:", pca_ckpt)

# Train IQL

In [ ]:
# Cell 5: train IQL with PCA-projected observations as latent states
iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=pca_encoder,
    repr_mode="pca",
    use_tqdm=False,
)

# Evaluation

In [ ]:
# Cell 6: evaluate the trained policy and save metrics
print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=pca_encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "latent_dim": LATENT_DIM,
        "pca_n_components": LATENT_DIM,
        "pca_checkpoint": str(pca_ckpt),
        "iql_epochs": EPOCHS,
        "obs_stats_path": str(OBS_DIR / "obs_stats.npz"),
        "iql_history": iql_history,
    },
)

print("Saved metrics:", metrics_path)
print(metrics)